# OpenFF Lipid Torsion Drives v4.0

This notebook generates additional torsiondrives from lipid-like molecules for Sage training 

In [1]:
import zstandard
import qcportal
import pathlib

from openff.toolkit import Molecule, ForceField
import numpy as np

from openff.qcsubmit.utils import get_symmetry_classes, get_symmetry_group
from openff.qcsubmit.workflow_components import TorsionIndexer
from openff.qcsubmit import workflow_components
from openff.qcsubmit.factories import TorsiondriveDatasetFactory
from openff.qcsubmit.utils.visualize import molecules_to_pdf

Matplotlib is building the font cache; this may take a moment.


In [2]:
def atom_weight(mol, idx):
    """Return atomic number (proxy for heaviness)."""
    return mol.atoms[idx].atomic_number


def torsion_score(mol, torsion):
    """Score torsion based on terminal atom heaviness (i and l)."""
    i, j, k, l = torsion
    return atom_weight(mol, i) + atom_weight(mol, l)


def load_molecules(
    files: list[str],
    target_params: list[str],
) -> list[Molecule]:
    """Load SMILES from files and assign dihedrals to rotate around any of the given target SMIRKS parameters
    Note: select BEST torsion per central bond (favor heavy atoms)"""
    
    case_molecules = []
    forcefield = ForceField("/media/julianne/DATA/Lipids/openff-lipid-ff-dev/force-fields/ff-iterations/openff-2.2.0.offxml")

    for file in files:
        molecules = Molecule.from_file(file, allow_undefined_stereo=True)
    
        for mol in molecules:
            # unique_central_bonds = set()
            torsion_indexer = TorsionIndexer()
            symmetry_classes = get_symmetry_classes(mol)

            # store BEST torsion per central bond
            best_torsions = {}  # key: (j,k), value: (torsion, score, symmetry_group)

            labels = forcefield.label_molecules(mol.to_topology())[0]["ProperTorsions"]

            for (i, j, k, l), parameter in labels.items():
                if parameter.id not in target_params:
                    continue
                central_bond = tuple(sorted([j, k]))
                # skip ring torsions
                if mol.get_bond_between(j, k).is_in_ring():
                    continue
                # skip duplicates
                # if central_bond in unique_central_bonds:
                #     continue

                symmetry_group = get_symmetry_group(central_bond, symmetry_classes)

                # compute score
                score = torsion_score(mol, (i, j, k, l))

                if central_bond in best_torsions:
                    _, existing_score, _ = best_torsions[central_bond]

                    if score > existing_score:
                        best_torsions[central_bond] = ((i, j, k, l), score, symmetry_group)
                else:
                    best_torsions[central_bond] = ((i, j, k, l), score, symmetry_group)

                torsion_indexer.add_torsion((i, j, k, l), symmetry_group, (-165, 180))
                # unique_central_bonds.add(central_bond)
    
            if len(torsion_indexer.torsions) == 0:
                continue  # skip molecule if no matching torsions found

            mol.properties["dihedrals"] = torsion_indexer
            case_molecules.append(mol)

    return case_molecules


def visualize(mols, filename):
    """Draw output molecules as PDF"""
    new_mols = []
    for mol in mols:
        for val in mol.properties["dihedrals"].torsions.values():
            new_mol = Molecule(mol)
            new_mol.properties["dihedrals"] = val.get_dihedrals
            new_mols.append(new_mol)
    molecules_to_pdf(new_mols, filename)

In [9]:
target_param_1 = ["t1"]

target_param_2 = ["t18", "t17"]

lipid_molecules_1 = load_molecules(files=["input.smi"], target_params=target_param_1)
len(lipid_molecules_1)

with open("input.smi", "r") as f:
    lines = f.readlines()[4:8]  # lines 5-8
    with open("subset.smi", "w") as temp_f:
        temp_f.writelines(lines)

# Load molecules from the temporary subset file
lipid_molecules_2 = load_molecules(files=["subset.smi"], target_params=target_param_2)
print(len(lipid_molecules_2))

# Combine both sets
lipid_molecules = lipid_molecules_1 + lipid_molecules_2
print(len(lipid_molecules))

[18:51:43] WARNING: no name column found on line 0
[18:51:43] WARNING: no name column found on line 1
[18:51:43] WARNING: no name column found on line 2
[18:51:43] WARNING: no name column found on line 3
[18:51:43] WARNING: no name column found on line 4
[18:51:43] WARNING: no name column found on line 5
[18:51:43] WARNING: no name column found on line 6
[18:51:43] WARNING: no name column found on line 7


4
12


[18:51:43] WARNING: no name column found on line 0
[18:51:43] WARNING: no name column found on line 1
[18:51:43] WARNING: no name column found on line 2
[18:51:43] WARNING: no name column found on line 3


In [10]:
visualize(lipid_molecules, "dataset.pdf")

# Dataset Preparation

In [ ]:
dataset_factory = TorsiondriveDatasetFactory()
dataset_factory.add_workflow_components(
    workflow_components.StandardConformerGenerator(max_conformers=5)
)

description = (
    '''A torsiondrive dataset was created to improve coverage of lipid-like headgroup, tail, and backbone parameters in the Sage force field. The model compounds were selected to enhance parameterization relevant to lipid force fields and include: 2-hexene, isopropylbutyrate, propylbutyrate, propylmethylphosphate, an esterified glycerol analogue ((M)EGLY), and an esterified glycerol-phosphate analogue ((M)PGLY). These molecules were manually selected based on CHARMM36 lipid FF training data.

The intended use of this dataset is to improve Sage parameters relevant to lipid simulations. The dataset provides minimal coverage for glycerol backbone parameters, ester parameters, alkene parameters, and amine and phosphate headgroup parameters. This effort builds on previous datasets to curate lipid-relevant QM data, including:
- 2024-07-17-OpenFF-Phosphate-Torsion-Drives-v1.0
- 2024-08-09-OpenFF-Alkane-Torsion-Drives-v1.0
- 2024-10-08-OpenFF-Lipid-Optimization-Training-Supplement-v1.0
- 2024-10-30-OpenFF-Lipid-Optimization-Benchmark-Supplement-v1.0
- 2025-07-25-OpenFF-Lipid-Torsion-Drives-v4.0

This dataset was generated at the B3LYP-D3BJ/DZVP level of theory using Psi4 with no implicit solvent and default specs. It includes molecules containing the elements C, H, O, and P, spanning formal charges of -1, 0. The molecular weights range up to 297.22 amu, with a mean of 194.30 amu.'''
)
dataset = dataset_factory.create_dataset(
    dataset_name="OpenFF Lipid Torsion Drives v5.0",
    tagline="Improve lipid torsiondrive coverage in Sage",
    description=description,
    molecules=lipid_molecules,
)

dataset.metadata.submitter = "JHoeflich1"
dataset.metadata.long_description_url = (
    "https://github.com/openforcefield/qca-dataset-submission/tree/master/"
    "submissions/" + str(pathlib.Path.cwd().name)
)

Preparation                   :  38%|████▉        | 3/8 [00:00<00:00, 21.39it/s][18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

Preparation                   :  75%|█████████▊   | 6/8 [00:00<00:00,  7.57it/s][18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:13] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) added/removed

[18:52:14] WARNING: Proton(s) 

In [12]:
# summarize dataset for readme
confs = np.array([len(mol.conformers) for mol in dataset.molecules])

print("* Number of unique molecules:", dataset.n_molecules)
# With multiple torsions per unique molecule, n_molecules * confs.mean() no
# longer equals the number of conformers. instead, the number of dihedrals *
# confs.mean() should equal the number of conformers. The dataset contains one
# record per driven torsion (rather than combining multiple dihedrals into the
# same record), so n_records is the same as manually adding up len(dihedrals)
# for each record.
print("* Number of driven torsions:", dataset.n_records)
print("* Number of filtered molecules:", dataset.n_filtered)
print("* Number of conformers:", sum(confs))
print(
    "* Number of conformers per molecule (min, mean, max): "
    f"{confs.min()}, {confs.mean():.2f}, {confs.max()}"
)

masses = [
    [
        sum([atom.mass.m for atom in molecule.atoms])
        for molecule in dataset.molecules
    ]
]
print(f"* Mean molecular weight: {np.mean(np.array(masses)):.2f}")
print(f"* Min molecular weight: {np.min(np.array(masses)):.2f}")
print(f"* Max molecular weight: {np.max(np.array(masses)):.2f}")
print("* Charges:", sorted(set(m.total_charge.m for m in dataset.molecules)))


print("## Metadata")
print(f"* Elements: {{{', '.join(dataset.metadata.dict()['elements'])}}}")


fields = [
    "basis",
    "implicit_solvent",
    "keywords",
    "maxiter",
    "method",
    "program",
]
for spec, obj in dataset.qc_specifications.items():
    od = obj.dict()
    print("* Spec:", spec)
    for field in fields:
        print(f"\t * {field}: {od[field]}")
    print("\t* SCF properties:")
    for field in od["scf_properties"]:
        print(f"\t\t* {field}")


# export the dataset
dataset.export_dataset("dataset.json.bz2")
dataset.molecules_to_file("output.smi", "smi")
dataset.visualize("dataset.pdf", columns=8)

* Number of unique molecules: 8
* Number of driven torsions: 27
* Number of filtered molecules: 0
* Number of conformers: 103
* Number of conformers per molecule (min, mean, max): 1, 3.81, 5
* Mean molecular weight: 225.06
* Min molecular weight: 84.16
* Max molecular weight: 297.22
* Charges: [-1.0, 0.0]
## Metadata
* Elements: {C, P, O, H}
* Spec: default
	 * basis: DZVP
	 * implicit_solvent: None
	 * keywords: {}
	 * maxiter: 200
	 * method: B3LYP-D3BJ
	 * program: psi4
	* SCF properties:
		* dipole
		* quadrupole
		* wiberg_lowdin_indices
		* mayer_indices
